In [1]:
import sys

import polars as pl
import torch


from modeling_module.data_loader.MultiPartDataModule import MultiPartDataModule
from modeling_module.data_loader.MultiPartExoDataModule import MultiPartExoDataModule

'''
pip3 install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128
https://developer.nvidia.com/cuda-12-8-0-download-archive
'''

MAC_DIR = '/Users/igwanhyeong/PycharmProjects/data_research/raw_data/'
WINDOW_DIR = 'C:/Users/USER/PycharmProjects/research/raw_data/'

if sys.platform == 'win32':
    DIR = WINDOW_DIR
    print(torch.cuda.is_available())
    print(torch.cuda.device_count())
    print(torch.version.cuda)
    print(torch.__version__)
    print(torch.cuda.get_device_name(0))
    print(torch.__version__)
else:
    DIR = MAC_DIR
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

save_dir = DIR + 'fit/20251115_LTB'

True
1
12.8
2.9.0.dev20250716+cu128
NVIDIA GeForce RTX 5080
2.9.0.dev20250716+cu128


In [3]:
target_dyn_demand_monthly = pl.read_parquet(DIR + 'target_dyn_demand_monthly.parquet').sort(['oper_part_no', 'demand_dt'])
target_dyn_demand_monthly = (target_dyn_demand_monthly
                                .group_by('oper_part_no', maintain_order = True)
                                .map_groups(lambda g: g.with_columns(pl.arange(1, len(g) + 1).alias('sequence')))
                            )


filtered_target = (target_dyn_demand_monthly
                    .group_by('oper_part_no')
                    .agg(pl.col('sequence').max().alias('sequence_max'))
                    .filter(pl.col('sequence_max') > 43)
                    .select('oper_part_no')
                   ) # seq Q75

target_dyn_demand_monthly = (target_dyn_demand_monthly
                                .join(filtered_target, on = 'oper_part_no', how = 'right')
                                .select(['oper_part_no', 'demand_dt', 'sequence', 'demand_qty'])
                             )
target_dyn_demand_monthly


oper_part_no,demand_dt,sequence,demand_qty
str,i64,i64,f64
"""C7220-61291""",201803,1,1.0
"""C7220-61291""",201806,2,10.0
"""C7220-61291""",201808,3,22.0
"""C7220-61291""",201809,4,13.0
"""C7220-61291""",201810,5,14.0
…,…,…,…
"""C7220-54102""",202603,46,1.0
"""C7220-54102""",202606,47,35.0
"""C7220-54102""",202607,48,1.0


In [9]:
target_dyn_demand = pl.read_parquet(DIR + 'parquets/dyn_demand.parquet').drop('part_no')

In [19]:
from resources.util.preprocess_util.demand_resampler import DemandResampler

resampler = DemandResampler(
    target_dyn_demand,
    name_col="oper_part_no",
    date_col="demand_dt",
    target_col="demand_qty",
    date_fmt="%Y%m%d",
)

# 1) 주간: Date + iso_yyyyww
weekly_df = resampler.to_weekly_filled(as_int=False, add_iso_yyyyww=True)
weekly_df

oper_part_no,demand_dt,demand_qty,iso_yyyyww
str,date,f64,i64
"""0001-1001""",2018-03-12,5.0,201811
"""0001-1001""",2018-03-19,0.0,201812
"""0001-1001""",2018-03-26,0.0,201813
"""0001-1001""",2018-04-02,0.0,201814
"""0001-1001""",2018-04-09,0.0,201815
…,…,…,…
"""ZZ90239""",2023-05-22,0.0,202321
"""ZZ90239""",2023-05-29,0.0,202322
"""ZZ90239""",2023-06-05,0.0,202323


In [38]:
from resources.util.preprocess_util.intermittent_demand_detector import IntermittentConfig, IntermittentDemandDetector

cfg = IntermittentConfig(
    name_col="oper_part_no",
    target_col="demand_qty",
    adi_threshold=1.32,
    cv2_threshold=0.49,
    count_threshold=0.0,   # 20 미만은 사실상 '거의 0'으로 간주
    use_cv2=True,         # ADI + CV^2 모두 사용
    min_periods=20,       # 20주 미만이면 간헐 판정 안 함(무조건 False)
)

detector = IntermittentDemandDetector(weekly_df, config=cfg)
detector.detect(return_stats=  True)

oper_part_no,is_sparsity,n_periods,n_zero,n_nz,zero_ratio,ADI,CV2
str,bool,u32,u32,u32,f64,f64,f64
"""76KD-0507""",true,235,0,235,0.0,1.0,27.823356
"""UD26-A0018A""",false,1,0,1,0.0,1.0,null
"""38365-2172-1""",true,312,0,312,0.0,1.0,19.245388
"""82EP-1790-2""",true,316,0,316,0.0,1.0,7.428844
"""CZ17-0027A""",true,67,0,67,0.0,1.0,23.470303
…,…,…,…,…,…,…,…
"""L79776""",true,129,0,129,0.0,1.0,71.21875
"""6212-600-022-00""",true,97,0,97,0.0,1.0,31.659722
"""L176900""",false,2,0,2,0.0,1.0,0.0


In [39]:
weekly_df.filter(pl.col('oper_part_no') == "UD26-A0018A")

oper_part_no,demand_dt,demand_qty,iso_yyyyww
str,date,f64,i64
"""UD26-A0018A""",2022-10-10,6.0,202241
